In [1]:
from pathlib import Path

ROOT_DIR      = Path().resolve().parent
REVIEW_PATH   = str(ROOT_DIR / "data" / "raw" / "Clothing_Shoes_and_Jewelry.jsonl.gz")
META_PATH     = str(ROOT_DIR / "data" / "raw" / "meta_Clothing_Shoes_and_Jewelry.jsonl.gz")
PROCESSED_DIR = str(ROOT_DIR / "data" / "processed")
REVIEW_RAW_PARQUET = str(ROOT_DIR / "data" / "raw" / "Clothing_Shoes_and_Jewelry.parquet")
META_RAW_PARQUET   = str(ROOT_DIR / "data" / "raw" / "meta_Clothing_Shoes_and_Jewelry.parquet")

In [2]:
from pyspark.sql import SparkSession
from pyspark.sql.types import *

# 1. Khởi tạo Spark Session nhẹ nhàng
spark = SparkSession.builder \
    .appName("Convert_Review_to_Parquet") \
    .master("local[*]") \
    .config("spark.driver.memory", "4g") \
    .getOrCreate()

print("🚀 Đang khởi động chuyển đổi file Review...")

# 2. Định nghĩa Schema chính xác để đọc nhanh nhất
review_schema = StructType([
    StructField("rating", FloatType(), True),
    StructField("title", StringType(), True),
    StructField("text", StringType(), True),
    StructField("user_id", StringType(), True),
    StructField("parent_asin", StringType(), True),
    StructField("timestamp", LongType(), True),
    StructField("helpful_vote", IntegerType(), True),
    StructField("verified_purchase", BooleanType(), True)
])


# 4. Đọc file JSON.gz và ghi ra Parquet ngay lập tức
print(f"📥 Đang đọc từ: {REVIEW_PATH}")
df_review = spark.read.schema(review_schema).json(REVIEW_PATH)

print(f"💾 Đang nén và ghi ra Parquet (Có thể mất vài phút)...")
# Dùng coalesce để gộp file lại, tránh sinh ra quá nhiều file rác
df_review.coalesce(12).write.mode("overwrite").parquet(REVIEW_RAW_PARQUET)

print(f"✅ Hoàn tất! File Review đã sẵn sàng tại: {REVIEW_RAW_PARQUET}")

🚀 Đang khởi động chuyển đổi file Review...
📥 Đang đọc từ: D:\Data\300_bai_code\300 bai code thieu nhi\amazon-clothing-analysis\data\raw\Clothing_Shoes_and_Jewelry.jsonl.gz
💾 Đang nén và ghi ra Parquet (Có thể mất vài phút)...
✅ Hoàn tất! File Review đã sẵn sàng tại: D:\Data\300_bai_code\300 bai code thieu nhi\amazon-clothing-analysis\data\raw\Clothing_Shoes_and_Jewelry.parquet


In [4]:
from pyspark.sql import SparkSession
from pyspark.sql.types import *

# Khởi tạo Spark Session (nếu chưa chạy code 1)
spark = SparkSession.builder \
    .appName("Convert_Meta_to_Parquet") \
    .master("local[*]") \
    .config("spark.driver.memory", "4g") \
    .getOrCreate()

print("\n🚀 Đang khởi động chuyển đổi file Metadata...")

# 1. Định nghĩa Schema cho Metadata
meta_schema = StructType([
    StructField("parent_asin", StringType(), True),
    StructField("title", StringType(), True),
    StructField("price", FloatType(), True),
    StructField("average_rating", FloatType(), True),
    StructField("rating_number", IntegerType(), True),
    StructField("main_category", StringType(), True),
    StructField("store", StringType(), True),
    StructField("description", ArrayType(StringType()), True)
])


# 3. Đọc JSON.gz và ghi ra Parquet
print(f"📥 Đang đọc từ: {META_PATH}")
df_meta = spark.read.schema(meta_schema).json(META_PATH)

print(f"💾 Đang nén và ghi ra Parquet...")
# File meta thường nhỏ hơn, chia 4 phần là đủ mượt
df_meta.coalesce(4).write.mode("overwrite").parquet(META_RAW_PARQUET)

print(f"✅ Hoàn tất! File Metadata đã sẵn sàng tại: {META_RAW_PARQUET}")


🚀 Đang khởi động chuyển đổi file Metadata...
📥 Đang đọc từ: D:\Data\300_bai_code\300 bai code thieu nhi\amazon-clothing-analysis\data\raw\meta_Clothing_Shoes_and_Jewelry.jsonl.gz
💾 Đang nén và ghi ra Parquet...
✅ Hoàn tất! File Metadata đã sẵn sàng tại: D:\Data\300_bai_code\300 bai code thieu nhi\amazon-clothing-analysis\data\raw\meta_Clothing_Shoes_and_Jewelry.parquet


In [7]:
df_meta_parquet = spark.read.parquet(REVIEW_RAW_PARQUET)
count_parquet = df_meta_parquet.count()
count_raw = df_review.count()

In [8]:
print(f"✅ Số dòng file gốc:    {count_raw:,}")
print(f"✅ Số dòng file Parquet: {count_parquet:,}")

✅ Số dòng file gốc:    66,033,346
✅ Số dòng file Parquet: 66,033,346


In [6]:
print(f"✅ Số dòng file gốc:    {count_raw:,}")
print(f"✅ Số dòng file Parquet: {count_parquet:,}")

✅ Số dòng file gốc:    7,218,481
✅ Số dòng file Parquet: 7,218,481
